In [0]:
%sql
CREATE CATALOG telecom_catalog_assign;
CREATE SCHEMA IF NOT EXISTS telecom_catalog_assign.landing_zone;
CREATE VOLUME IF NOT EXISTS telecom_catalog_assign.landing_zone.landing_vol;




In [0]:
dbutils.fs.mkdirs('/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/')
dbutils.fs.mkdirs('/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/')
dbutils.fs.mkdirs('/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/')

In [0]:
customer_csv = ''' 101,Arun,31,Chennai,PREPAID 102,Meera,45,Bangalore,POSTPAID 103,Irfan,29,Hyderabad,PREPAID 104,Raj,52,Mumbai,POSTPAID 105,,27,Delhi,PREPAID 106,Sneha,abc,Pune,PREPAID '''
customer_csv = customer_csv.replace(" ","\n")
dbutils.fs.put('/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv', customer_csv, True)
usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count 101\t320\t1500\t20 102\t120\t4000\t5 103\t540\t600\t52 104\t45\t200\t2 105\t0\t0\t0 '''
usage_tsv = usage_tsv.replace(" ","\n")
dbutils.fs.put('/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv', usage_tsv, True)
import re
tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp 5001|101|TWR01|-80|2025-01-10 10:21:54 5004|104|TWR05|-75|2025-01-10 11:01:12 '''
usage_log= re.sub(r' (?=\d+\|)', '\n', tower_logs_region1)
dbutils.fs.put('/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower.log', usage_log, True)


In [0]:
dbutils.fs.ls('/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/')
dbutils.fs.ls('/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/')
dbutils.fs.ls('/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/')

In [0]:
df1 = spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/",header=True,recursiveFileLookup=True,pathGlobFilter="*.log",sep='|')
df1.display()


In [0]:
df1 = spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/",header=True,inferSchema=True,pathGlobFilter="*.log",sep='|')
df1.display()

In [0]:
df1 = spark.read.csv(path=["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv","/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/custs - Copy.csv"],sep=',',inferSchema=True).toDF("id","name","age","gender","address")
df1.display()

In [0]:
df1 = spark.read.option("recursiveFileLookup","true").csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/",inferSchema=True,sep=',').toDF("id","name","age","gender","address")
df1.display()

In [0]:
df1 = spark.read.options("recursiveFileLookup","true").csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/",header=False,sep=',')
df1.display()

In [0]:
##if header is True the first row is assigned as header of the table
##if inferschema is True the data type is detected automatically assigned based on the value
df1 = spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",header=True,inferSchema=True,sep=',')
df1.display()
df2 = spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv",header=True,inferSchema=True,sep='\t')
df2.display()
df3 = spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower.log",header=True,inferSchema=True,sep='|')
df3.display()


In [0]:
##if header is false the default column names are _c0,_c1,_c2,_c3,_c4 names are assigned
##if inferschema is false the default data type is string
df1 = spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",header=False,inferSchema=False,sep=',')
df1.display()
df2 = spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv",header=False,inferSchema=False,sep='\t')
df2.display()

In [0]:
df1 = spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",inferSchema=True,sep=',').toDF("id","name","age","city","plan")
df1.display()

In [0]:
schema = "cust_id INT, vmins INT, data_mb INT, sms_count INT"
df2 = spark.read.schema(schema).csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv",sep='\t')
df2.display()

In [0]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType,DateType,TimestampType
schema = StructType([StructField("event_id", StringType(), True),StructField("custid", IntegerType(), True),StructField("towerid", StringType(), True),StructField("signalstrength", IntegerType(), True),StructField("DisplayedDate", TimestampType(), True)])
df = spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower.log", schema=schema,sep='|')
df.display()

In [0]:
df1 = spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower.log",inferSchema=True,sep='|')
df1.display()


In [0]:
import re

tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp 5001|101|TWR01|-80|2025-01-10 10:21:54 5004|104|TWR05|-75|2025-01-10 11:01:12'''

# Add newline before each record starting with digits + pipe
tower_logs_region1_fixed = re.sub(r' (?=\d+\|)', '\n', tower_logs_region1)

log_fixed = tower_logs_region1_fixed.replace(" ","|")

print(tower_logs_region1_fixed)